In [12]:
pip install transformers datasets evaluate seqeval torch -qqq

In [13]:
import datasets
import transformers

datasets.disable_progress_bar()
transformers.utils.logging.disable_progress_bar()

In [14]:

from tqdm import tqdm

for i in tqdm(range(100), disable=True):
    pass

In [15]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer
)
import evaluate
import numpy as np

from datasets import load_dataset

# Load the Dataset
dataset=load_dataset("lhoestq/conll2003")
chunk_labels=[
    "O", "B-ADJP", "I-ADJP", "B-ADVP", "I-ADVP", "B-CONJP", "I-CONJP",
    "B-INTJ", "I-INTJ", "B-LST", "I-LST", "B-NP", "I-NP", "B-PP", "I-PP",
    "B-PRT", "I-PRT", "B-SBAR", "I-SBAR", "B-UCP", "I-UCP", "B-VP", "I-VP"
]

id2label={i: label for i, label in enumerate(chunk_labels)}
label2id={label: i for i, label in enumerate(chunk_labels)}
print("Labels successfully mapped!")

# Load Tokenizer & DistilBERT
model_checkpoint="distilbert-base-cased"
tokenizer=AutoTokenizer.from_pretrained(model_checkpoint)
model=AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(chunk_labels),
    id2label=id2label,
    label2id=label2id
)

# Data Preprocessing
def tokenize_and_align_labels(examples):
    tokenized_inputs=tokenizer(
        examples["tokens"], truncation=True, is_split_into_words=True
    )
    labels=[]
    for i, label in enumerate(examples["chunk_tags"]):
        word_ids=tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx=None
        label_ids=[]
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx=word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"]=labels
    return tokenized_inputs
tokenized_datasets=dataset.map(tokenize_and_align_labels, batched=True)

Labels successfully mapped!


DistilBertForTokenClassification LOAD REPORT from: distilbert-base-cased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [16]:
seqeval=evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions=np.argmax(predictions, axis=2)
    true_predictions=[
        [chunk_labels[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels=[
        [chunk_labels[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    results=seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }
data_collator=DataCollatorForTokenClassification(tokenizer=tokenizer)

args=TrainingArguments(
    output_dir="./distilbert-chunking",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
)

trainer=Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=tokenizer,
)
trainer.train()


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.417730,0.184271,0.910851,0.903136,0.906977,0.951482
2,0.148125,0.162177,0.919751,0.912461,0.916091,0.956369
3,0.117075,0.159694,0.920278,0.914986,0.917625,0.957108


TrainOutput(global_step=2634, training_loss=0.19520270543833437, metrics={'train_runtime': 280.5376, 'train_samples_per_second': 150.151, 'train_steps_per_second': 9.389, 'total_flos': 525452463338040.0, 'train_loss': 0.19520270543833437, 'epoch': 3.0})

In [17]:
from transformers import pipeline
chunking_pipeline=pipeline("token-classification", model=model, tokenizer=tokenizer, aggregation_strategy="simple")

text="John works at Google in California"
predictions=chunking_pipeline(text)

print("Input:", text)
for entity in predictions:
    print(f"Word/Phrase: {entity['word']} | Tag: {entity['entity_group']} | Score: {entity['score']:.4f}")

Input: John works at Google in California
Word/Phrase: John | Tag: NP | Score: 0.9989
Word/Phrase: works | Tag: VP | Score: 0.7864
Word/Phrase: at | Tag: PP | Score: 0.9970
Word/Phrase: Google | Tag: NP | Score: 0.9969
Word/Phrase: in | Tag: PP | Score: 0.9979
Word/Phrase: California | Tag: NP | Score: 0.9982


## Final Report: Token Classification (Chunking) Project

This project successfully implemented and demonstrated a token classification model for chunking, leveraging the Hugging Face Transformers library and the `lhoestq/conll2003` dataset.

### Project Goal
The primary goal was to train a DistilBERT-based model to perform chunking, which involves grouping words into meaningful phrases (e.g., Noun Phrases, Verb Phrases) within a sentence.

### Key Steps and Implementation Details

1.  **Environment Setup**: Necessary libraries, including `transformers`, `datasets`, `evaluate`, `seqeval`, and `torch`, were installed.
2.  **Dataset Loading**: The `lhoestq/conll2003` dataset was loaded, specifically utilizing its `chunk_tags` for the token classification task. Custom `id2label` and `label2id` mappings were created for the 23 distinct chunk labels.
3.  **Model and Tokenizer Initialization**: The `distilbert-base-cased` tokenizer and `AutoModelForTokenClassification` were loaded. The model was initialized with the defined `num_labels`, `id2label`, and `label2id` to ensure proper classification output.
4.  **Data Preprocessing**: A crucial `tokenize_and_align_labels` function was developed. This function handled subword tokenization (a common challenge with Transformers) by ensuring that only the first subword of a token received its corresponding label, while subsequent subwords were assigned an ignored label (`-100`) to prevent interference with loss calculation. This ensured accurate alignment between tokens and their chunk labels.
5.  **Model Training Setup**: The `seqeval` library was loaded to compute standard metrics (precision, recall, F1, accuracy) for token classification. `DataCollatorForTokenClassification` was used to prepare batches dynamically, padding sequences to the longest length in a batch. `TrainingArguments` were configured for 3 epochs, a learning rate of 2e-5, and a batch size of 16.
6.  **Model Training**: The `Trainer` API was used to fine-tune the DistilBERT model on the `tokenized_datasets['train']` and `tokenized_datasets['validation']`. The training concluded successfully with a reported training loss of approximately 0.195.
7.  **Inference and Demonstration**: A `pipeline` for 'token-classification' was created using the fine-tuned model and tokenizer, with an `aggregation_strategy='simple'` to combine subword predictions into single entities. A sample text, "John works at Google in California," was processed, and the pipeline accurately identified and classified phrases like "John" (NP), "works" (VP), "at" (PP), "Google" (NP), "in" (PP), and "California" (NP), demonstrating the model's effectiveness.

### Challenges Encountered and Resolutions

*   **Deprecated Dataset Scripts**: Initial attempts to load `conll2003` faced `RuntimeError` due to deprecated dataset scripts. This was resolved by switching to `lhoestq/conll2003` and ensuring library versions were compatible.
*   **Library Version Incompatibilities**: `StrictDataclassDefinitionError` and other issues arose due to discrepancies between `transformers` and `huggingface_hub` versions. These were primarily addressed by upgrading the libraries and performing necessary runtime restarts to ensure the new versions were correctly loaded.
*   **NameError**: A `NameError` (`Model` instead of `model`) during `Trainer` initialization was identified and corrected.

### Observations and Insights

*   **Subword Tokenization and Label Alignment**: The project successfully navigated the complexities of subword tokenization by implementing careful label alignment, which is critical for accurate token classification with transformer models.
*   **`seqeval` Metric**: The use of `seqeval` proved essential for evaluating the model's performance, as it correctly assesses predicted sequences rather than individual token accuracy.
*   **DistilBERT Efficiency**: Fine-tuning DistilBERT yielded strong results, highlighting its efficiency as a capable model for local sequence labeling tasks even with relatively small datasets.
*   **Importance of Environment Management**: The frequent need for runtime restarts after library upgrades emphasized the importance of robust environment management in Colab to ensure changes are effectively applied.

This project successfully demonstrated the end-to-end process of building and evaluating a chunking model, providing a solid foundation for further natural language understanding tasks.